In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


#-------------------------------------------Tools-----------------------------------------\
from tools_local.speak import speak_text



    
import subprocess
@tool
def open_task_manager() -> str:
    """
    Open the Windows Task Manager.

    Use this tool whenever the user asks to:
    - open task manager
    - launch task manager
    - show task manager
    - monitor running processes
    - check CPU usage
    - check RAM usage
    - check GPU usage
    """
    try:
        subprocess.Popen("taskmgr")
        return "Task Manager opened successfully."
    except Exception as e:
        return f"Failed to open Task Manager.\n{e}"

#--------------------------------------------End-----------------------------------------/

# Using Tool supported chatmodel LLm from ollama Api
llm = ChatOllama(model="gemma4:31b-cloud",temperature=0,)
tools = [open_task_manager,show_notification,speak_text] # for Tool Registration for Agent

llm = llm.bind_tools(tools) # Agent tool connection


class State(TypedDict): 
    messages: Annotated[list, add_messages]


def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return {
        "messages": [response]
    }



graph = StateGraph(State)
graph.add_node("chatbot", chatbot)
graph.add_node("tools",ToolNode(tools))
graph.add_edge(START,"chatbot")
graph.add_conditional_edges("chatbot",tools_condition,)
graph.add_edge("tools","chatbot")
app = graph.compile()


print("Jarvis Started")
print("Type 'close' to quit.\n")

while True:
    user_input = input("You : ")

    if user_input.lower() == "close":
        break

    print("\nJarvis : ", end="", flush=True)
    send_to_llm = app.stream({"messages": [HumanMessage(user_input)]},stream_mode="messages",)
    
    for message, metadata in send_to_llm:
        if hasattr(message, "content") and message.content:
            print(message.content, end="", flush=True)

    print("\n")

Jarvis Started
Type 'close' to quit.


Jarvis : Hello! How can I help you today?


Jarvis : Speech started successfully.
Audio File: C:\Users\Abhishek\AppData\Local\Temp\tmpnpupvazj.mp3I am your AI assistant, designed to help you manage your computer, automate tasks, and provide information quickly and efficiently. I can interact with your system to open applications, send notifications, and assist you with various digital workflows to make your experience smoother. 

How can I help you today?


Jarvis : Task Manager opened successfully.Maine Task Manager khol diya hai. Ab aap wahan CPU, RAM aur running processes dekh sakte hain. Aapko specifically kya check karna hai?


Jarvis : Speech started successfully.
Audio File: C:\Users\Abhishek\AppData\Local\Temp\tmpj2ukj1_z.mp3Ji haan, main bol kar bata sakta hoon. Aap kya jaanna chahte hain?



In [ ]:
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma4:31b-cloud",
    temperature=0,
)

print("Jarvis Started\n")

while True:

    q = input("You : ")

    if q.lower() == "exit":
        break

    print("\nJarvis : ", end="", flush=True)

    for chunk in llm.stream(
        [
            HumanMessage(content=q)
        ]
    ):
        print(chunk.content, end="", flush=True)

    print("\n")

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list, add_messages]


llm = ChatOllama(model="gemma4:31b-cloud",temperature=0,)


def chatbot(state: State):
    return {
        "messages": [llm.invoke(state["messages"])]
    }


graph = StateGraph(State)

graph.add_node("chatbot", chatbot)

graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()


while True:

    q = input("You : ")

    if q.lower() == "exit":
        break

    result = app.invoke(
        {
            "messages": [
                HumanMessage(content=q)
            ]
        }
    )

    print("AI :", result["messages"][-1].content)

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0,
)


for chunk in app.stream(
    {
        "messages": [
            HumanMessage(content=q)
        ]
    }
):
    print(chunk)
    

for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)